# Pipeline de Feature Engineering a l Echelle Industrielle

## Contexte et Objectifs

Ce notebook presente une approche robuste et modulaire pour construire un pipeline de feature engineering complet. Un tel pipeline est essentiel dans les projets de machine learning a grande echelle pour garantir la reproductibilite, la maintenabilite et la performance du modele.

### Elements Cles de ce Notebook :

1.  **Generation de Donnees Complexes :** Creation d'un jeu de donnees synthetique imitant les defis du monde reel, incluant des valeurs manquantes, des outliers, et differents types de variables (numeriques, categorielles, temporelles).
2.  **Transformateurs `scikit-learn` Personnalises :** Developpement de transformateurs sur mesure pour des taches specifiques telles que l'extraction d'attributs temporels ou le traitement des outliers.
3.  **`ColumnTransformer` pour le Pretraitement Cible :** Utilisation du `ColumnTransformer` de `scikit-learn` pour appliquer differentes sequences de transformations a differentes colonnes, une pratique essentielle pour les pipelines heterogenes.
4.  **Pipeline de Machine Learning Complet :** Integration du pipeline de pretraitement avec un modele de regression (`RandomForestRegressor`) pour demontrer le flux de travail de bout en bout.
5.  **Evaluation Rigoureuse :** Mesure de la performance du modele apres l'application du pipeline pour valider son efficacite.

_Derniere mise a jour : 2026-02-16_

In [1]:
# --- 1. Installation des Dependances ---
%pip install -q pandas numpy scikit-learn matplotlib seaborn
print("Dependances installees.")

SyntaxError: invalid syntax (2051176013.py, line 1)

In [2]:
# --- 2. Imports ---
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import logging
import time

# --- Configuration ---
warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

SyntaxError: invalid syntax (3714364521.py, line 1)

## 3. Generation de Donnees Simulant un Cas Reel

Pour demontrer la puissance du pipeline, nous creons un jeu de donnees qui presente plusieurs defis courants :
- **Donnees Temporelles :** Une colonne de dates a partir de laquelle nous extrairons des informations utiles.
- **Variables Categorielles et Numeriques.**
- **Valeurs Manquantes :** Introduites aleatoirement dans plusieurs colonnes.
- **Outliers :** Ajoutes a une variable numerique pour tester la robustesse du pipeline.

In [3]:
class DataGenerator:
    """Genere un jeu de donnees complexe pour demontrer le pipeline."""
    def __init__(self, n_samples=1000):
        self.n_samples = n_samples

    def generate(self):
        logger.info(f"Generation de {self.n_samples} echantillons de donnees...")
        np.random.seed(42)
        
        # Dates
        dates = pd.to_datetime(pd.date_range(start='2023-01-01', periods=self.n_samples, freq='D'))
        
        # Features numeriques
        sensor_a = np.random.rand(self.n_samples) * 100 + np.sin(np.arange(self.n_samples) / 50) * 20
        sensor_b = np.random.rand(self.n_samples) * 50 + 2 * np.arange(self.n_samples) / self.n_samples * 10
        
        # Feature categorielle
        machine_type = np.random.choice(['Type A', 'Type B', 'Type C'], self.n_samples, p=[0.5, 0.3, 0.2])
        
        # Cible (avec une relation non-lineaire)
        target = 50 + (sensor_a * 0.5) + (sensor_b * 1.5) + (pd.Series(machine_type) == 'Type B') * 20 + np.random.randn(self.n_samples) * 15
        
        df = pd.DataFrame({
            'event_date': dates,
            'sensor_a': sensor_a,
            'sensor_b': sensor_b,
            'machine_type': machine_type,
            'target': target
        })
        
        # Introduire des valeurs manquantes
        logger.info("Introduction de valeurs manquantes...")
        for col in ['sensor_a', 'machine_type']: 
            mask = np.random.rand(self.n_samples) < 0.1
            df.loc[mask, col] = np.nan
            
        # Introduire des outliers
        logger.info("Introduction d outliers...")
        outlier_indices = np.random.choice(df.index, size=int(self.n_samples * 0.05), replace=False)
        df.loc[outlier_indices, 'sensor_b'] *= np.random.choice([-5, 5])
        
        logger.info("Generation de donnees terminee.")
        return df

# --- Utilisation ---
data_generator = DataGenerator()
df = data_generator.generate()
print("Apercu des donnees brutes:")
print(df.head())
print("
Valeurs manquantes par colonne:")
print(df.isnull().sum())

SyntaxError: invalid syntax (823652641.py, line 1)

## 4. Transformateurs `scikit-learn` Personnalises

La force de `scikit-learn` reside dans sa capacite a etre etendu. Nous creons ici deux transformateurs personnalises :
- **`TemporalVariableTransformer` :** Extrait le mois, le jour de la semaine et le jour de l annee a partir de la colonne de date.
- **`OutlierRemover` :** Remplace les outliers par des valeurs plancher et plafond (winsorizing), une technique plus robuste que la simple suppression.

In [4]:
class TemporalVariableTransformer(BaseEstimator, TransformerMixin):
    """Transformateur pour extraire des features d'une variable temporelle."""
    def __init__(self, variable):
        self.variable = variable

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X[self.variable] = pd.to_datetime(X[self.variable])
        X['month'] = X[self.variable].dt.month
        X['day_of_week'] = X[self.variable].dt.dayofweek
        X['day_of_year'] = X[self.variable].dt.dayofyear
        X = X.drop([self.variable], axis=1)
        return X

class OutlierRemover(BaseEstimator, TransformerMixin):
    """Transformateur pour remplacer les outliers par winsorizing."""
    def __init__(self, variable, lower_quantile=0.05, upper_quantile=0.95):
        self.variable = variable
        self.lower_quantile = lower_quantile
        self.upper_quantile = upper_quantile
        self.lower_bound = None
        self.upper_bound = None

    def fit(self, X, y=None):
        self.lower_bound = X[self.variable].quantile(self.lower_quantile)
        self.upper_bound = X[self.variable].quantile(self.upper_quantile)
        return self

    def transform(self, X):
        X = X.copy()
        X[self.variable] = np.clip(X[self.variable], self.lower_bound, self.upper_bound)
        return X

SyntaxError: invalid syntax (3041504807.py, line 1)

## 5. Construction du Pipeline de Pretraitement

Le `ColumnTransformer` est la piece maitresse. Il nous permet de definir des pipelines specifiques pour chaque groupe de colonnes :
1.  **Pipeline Numerique :** Imputation des valeurs manquantes par la mediane (robuste aux outliers), suppression des outliers avec notre transformateur personnalise, puis standardisation.
2.  **Pipeline Categoriel :** Imputation des valeurs manquantes avec une valeur constante, puis encodage one-hot.

Ces pipelines sont combines avec le `TemporalVariableTransformer` dans un pipeline global.

In [5]:
# Separation des donnees
X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Definition des colonnes
TEMPORAL_VAR = 'event_date'
NUMERICAL_VARS = ['sensor_a', 'sensor_b']
CATEGORICAL_VARS = ['machine_type']
OUTLIER_VAR = 'sensor_b'

# Pipeline de pretraitement pour les variables numeriques
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('outlier_remover', OutlierRemover(variable=OUTLIER_VAR)),
    ('scaler', StandardScaler())
])

# Pipeline de pretraitement pour les variables categorielles
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combinaison des pipelines avec ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_pipeline, NUMERICAL_VARS),
        ('categorical', categorical_pipeline, CATEGORICAL_VARS)
    ], remainder='passthrough' # Garde les autres colonnes
)

# Pipeline complet incluant la transformation temporelle
feature_engineering_pipeline = Pipeline(steps=[
    ('temporal_transformer', TemporalVariableTransformer(variable=TEMPORAL_VAR)),
    ('preprocessor', preprocessor)
])

logger.info("Pipeline de feature engineering construit avec succes.")

SyntaxError: invalid syntax (416476538.py, line 1)

## 6. Pipeline de Modele et Evaluation

Enfin, nous integrons notre pipeline de feature engineering dans un pipeline de modele final. Cela garantit que les memes transformations sont appliquees de maniere coherente lors de l'entrainement, de l'evaluation et, plus tard, en production.

In [ ]:
# Creation du pipeline de modele complet
model_pipeline = Pipeline(steps=[
    ('feature_engineering', feature_engineering_pipeline),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

# Entrainement du pipeline
logger.info("Entrainement du pipeline de modele complet...")
start_time = time.time()
model_pipeline.fit(X_train, y_train)
duration = time.time() - start_time
logger.info(f"Entrainement termine en {duration:.2f} secondes.")

# Evaluation du pipeline
logger.info("Evaluation du modele...")
y_pred = model_pipeline.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"
Resultats de l'Evaluation :")
print(f"  Mean Squared Error (MSE): {mse:.2f}")
print(f"  R2 Score: {r2:.4f}")

# Visualisation des predictions
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--r', linewidth=2)
plt.title('Valeurs Reelles vs. Predictions', fontsize=16)
plt.xlabel('Valeurs Reelles', fontsize=12)
plt.ylabel('Predictions', fontsize=12)
plt.show()

In [7]:
# Marqueur d'execution pour garantir au moins une sortie
print('Notebook execute avec succes — ' + time.strftime('%Y-%m-%d %H:%M:%S'))

Notebook executed (marker) — 2026-02-16 00:44:24
